# Catálogo de Dados — Camada Gold

Documentação das tabelas e colunas do modelo dimensional (esquema estrela), registrada como comentários no Unity Catalog. Descreve o significado, tipo, unidade e domínio de valores de cada campo.

In [0]:
%sql
-- Comentário da tabela fato_vendas
COMMENT ON TABLE ecommerce_br.gold.fato_vendas IS
'Tabela fato do esquema estrela, no grão de item de pedido. Cada linha representa um item vendido, com métricas de valor (preço, frete), entrega (prazo, atraso) e satisfação (nota da avaliação). Chaves ligam às dimensões de produto, cliente, vendedor e tempo.';

In [0]:
%sql
-- Comentários das colunas do fato_vendas
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN order_id COMMENT 'Identificador do pedido. Chave de ligação com o pedido de origem.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN order_item_id COMMENT 'Sequencial do item dentro do pedido (1, 2, 3...). Um pedido pode ter vários itens.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN product_id COMMENT 'Identificador do produto. Chave de ligação com dim_produto.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN seller_id COMMENT 'Identificador do vendedor. Chave de ligação com dim_vendedor.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN customer_id COMMENT 'Identificador do cliente no pedido. Chave de ligação com dim_cliente.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN order_status COMMENT 'Status do pedido (delivered, shipped, canceled, unavailable, invoiced, processing, created, approved).';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN order_purchase_timestamp COMMENT 'Data e hora da compra. Liga-se à dim_tempo pela data.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN price COMMENT 'Preço do item em reais (decimal, 2 casas). Não inclui frete.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN freight_value COMMENT 'Valor do frete do item em reais (decimal, 2 casas).';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN prazo_entrega_dias COMMENT 'Dias entre a compra e a entrega ao cliente. Nulo para pedidos não entregues.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN atraso_dias COMMENT 'Dias entre a entrega real e a prevista. Positivo = atrasado; negativo = adiantado; nulo para não entregues.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN review_score COMMENT 'Nota da avaliação do pedido (1 a 5). Repete-se entre itens do mesmo pedido; nula se o pedido não foi avaliado.';
ALTER TABLE ecommerce_br.gold.fato_vendas ALTER COLUMN data_processamento_gold COMMENT 'Timestamp de processamento da camada Gold (linhagem).';

In [0]:
%sql
-- Catálogo da dim_produto
COMMENT ON TABLE ecommerce_br.gold.dim_produto IS
'Dimensão de produtos. Descreve cada produto com sua categoria (em português e inglês) e atributos físicos. Chave: product_id.';

ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_id COMMENT 'Identificador único do produto. Chave da dimensão.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_category_name COMMENT 'Categoria do produto em português. "indefinida" quando ausente na origem.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_category_name_english COMMENT 'Categoria do produto em inglês (traduzida). "undefined" quando sem tradução.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_weight_g COMMENT 'Peso do produto em gramas. Nulo quando não informado.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_length_cm COMMENT 'Comprimento do produto em centímetros. Nulo quando não informado.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_height_cm COMMENT 'Altura do produto em centímetros. Nulo quando não informado.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN product_width_cm COMMENT 'Largura do produto em centímetros. Nulo quando não informado.';
ALTER TABLE ecommerce_br.gold.dim_produto ALTER COLUMN data_processamento_gold COMMENT 'Timestamp de processamento da camada Gold (linhagem).';

In [0]:
%sql
-- Catálogo da dim_vendedor
COMMENT ON TABLE ecommerce_br.gold.dim_vendedor IS
'Dimensão de vendedores. Descreve cada vendedor com localização e coordenadas geográficas (para cálculo de distância). Chave: seller_id.';

ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_id COMMENT 'Identificador único do vendedor. Chave da dimensão.';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_zip_code_prefix COMMENT 'Prefixo do CEP do vendedor (texto, preserva zeros à esquerda).';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_city COMMENT 'Cidade do vendedor (padronizada em maiúsculas).';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_state COMMENT 'Sigla do estado (UF) do vendedor.';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_lat COMMENT 'Latitude média do CEP do vendedor. Nula para CEPs sem geolocalização.';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN seller_lng COMMENT 'Longitude média do CEP do vendedor. Nula para CEPs sem geolocalização.';
ALTER TABLE ecommerce_br.gold.dim_vendedor ALTER COLUMN data_processamento_gold COMMENT 'Timestamp de processamento da camada Gold (linhagem).';

In [0]:
%sql
-- Catálogo da dim_cliente
COMMENT ON TABLE ecommerce_br.gold.dim_cliente IS
'Dimensão de clientes. Descreve cada cliente com localização e coordenadas geográficas. Mantém os dois identificadores: customer_id (por pedido) e customer_unique_id (pessoa real). Chave: customer_id.';

ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_id COMMENT 'Identificador do cliente no pedido. Chave da dimensão (liga ao fato).';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_unique_id COMMENT 'Identificador único da pessoa. Compras diferentes do mesmo cliente compartilham este id.';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_zip_code_prefix COMMENT 'Prefixo do CEP do cliente (texto, preserva zeros à esquerda).';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_city COMMENT 'Cidade do cliente (padronizada em maiúsculas).';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_state COMMENT 'Sigla do estado (UF) do cliente.';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_lat COMMENT 'Latitude média do CEP do cliente. Nula para CEPs sem geolocalização.';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN customer_lng COMMENT 'Longitude média do CEP do cliente. Nula para CEPs sem geolocalização.';
ALTER TABLE ecommerce_br.gold.dim_cliente ALTER COLUMN data_processamento_gold COMMENT 'Timestamp de processamento da camada Gold (linhagem).';

In [0]:
%sql
-- Catálogo da dim_tempo
COMMENT ON TABLE ecommerce_br.gold.dim_tempo IS
'Dimensão de tempo. Uma linha por data de compra, desdobrada em componentes para análises temporais. Chave: data_compra.';

ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN data_compra COMMENT 'Data da compra (sem hora). Chave da dimensão.';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN ano COMMENT 'Ano da compra (ex.: 2017).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN mes COMMENT 'Mês da compra (1 a 12).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN dia COMMENT 'Dia do mês da compra (1 a 31).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN trimestre COMMENT 'Trimestre do ano (1 a 4).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN nome_mes COMMENT 'Nome do mês por extenso, em inglês (ex.: September).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN dia_semana COMMENT 'Dia da semana (1 = domingo a 7 = sábado).';
ALTER TABLE ecommerce_br.gold.dim_tempo ALTER COLUMN data_processamento_gold COMMENT 'Timestamp de processamento da camada Gold (linhagem).';